# Yavatmal Revenue-Circle — Satellite Feature Extraction

Extracts Kharif-season (Jun–Oct) satellite & climate features for **110 revenue circle centroids** in Yavatmal district.  
These centroids come from the scraped PMFBY IU-level data (`pmfby_yavatmal_iu_kharif.csv`) geocoded via Nominatim.

**Target variable:** `claim_ratio` = Total Claim Paid / Sum Insured (from PMFBY IU data)  
**Seasons:** Kharif 2021–2025 (Jun–Oct each year)  
**Buffer per point:** 5 km radius circle  

**Features extracted per site × year:**
| Feature | Source | Notes |
|---|---|---|
| SPI30 (30-day SPI) | CHIRPS daily | Standardised rain anomaly |
| cumRain_mm | CHIRPS daily | Total kharif rainfall |
| drySpellDays | CHIRPS daily | Days with <2 mm in 30-day window |
| tmax_anom | ERA5-Land | Mean anomaly vs site 5-yr baseline |
| SM_anom | SMAP / ERA5 SM | Volumetric soil moisture anomaly |
| NDVI_peak | MODIS MOD13Q1 | Peak NDVI in kharif |
| NDVI_anom | MODIS MOD13Q1 | Anomaly vs site 5-yr median |
| VHI_mean | MODIS (NDVI+LST) | Vegetation Health Index |
| LST_anom | MODIS MOD11A2 | LST anomaly |

**Output:** `yavatmal_rc_features.csv`  — one row per (revenue_circle, year)

---
*Run cells top-to-bottom. GEE authentication required on first run.*

In [ ]:
import ee
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
GEE_PROJECT = 'earth-mrv'
try:
    ee.Initialize(project=GEE_PROJECT)
    print('GEE initialised.')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE authenticated + initialised.')

In [ ]:
# ── Load geocoded revenue circle centroids ────────────────────────────────────
coords_df = pd.read_csv('data/processed/yavatmal_rc_coords.csv')
# Drop any that failed geocoding
coords_df = coords_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f'Revenue circles with coordinates: {len(coords_df)}')
print(f'Geocode flag breakdown: {coords_df["geocode_flag"].value_counts().to_dict()}')
coords_df.head()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
KHARIF_YEARS = [2021, 2022, 2023, 2024, 2025]
# Kharif: June 1 → October 31
KHARIF_START_MD = '06-01'
KHARIF_END_MD   = '10-31'

BUFFER_M = 5000   # 5 km radius around centroid
SCALE_M  = 1000   # GEE reducer scale (1 km)

OUT_CSV = Path('data/processed/yavatmal_rc_features_v1.csv')
print(f'Processing {len(coords_df)} sites × {len(KHARIF_YEARS)} years = {len(coords_df)*len(KHARIF_YEARS)} cell-years')

In [ ]:
# ── Compute per-pixel baselines for anomaly features ──────────────────────────
# ERA5 tmax baseline (2016-2025 Jun-Oct mean)
era5_base = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
               .select('temperature_2m_max')
               .filter(ee.Filter.calendarRange(6, 10, 'month'))
               .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
               .mean())

# MODIS NDVI baseline (2016-2025 kharif median)
modis_ndvi_base = (ee.ImageCollection('MODIS/061/MOD13Q1')
                     .select('NDVI')
                     .filter(ee.Filter.calendarRange(6, 10, 'month'))
                     .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                     .median()
                     .multiply(0.0001))

# MODIS LST baseline
modis_lst_base  = (ee.ImageCollection('MODIS/061/MOD11A2')
                     .select('LST_Day_1km')
                     .filter(ee.Filter.calendarRange(6, 10, 'month'))
                     .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                     .mean()
                     .multiply(0.02).subtract(273.15))

print('Baseline images prepared.')

In [ ]:
# ── GEE helper: compute SPI from CHIRPS ──────────────────────────────────────
def chirps_spi30(start_date, end_date):
    """Return 30-day rolling precipitation anomaly image (Z-score)."""
    chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .select('precipitation')
                .filter(ee.Filter.date(
                    ee.Date(start_date).advance(-60, 'day'), end_date)))

    def rolling30(img):
        d = img.date()
        return (chirps.filter(ee.Filter.date(d.advance(-29, 'day'), d.advance(1, 'day')))
                      .sum()
                      .rename('roll30')
                      .set('system:time_start', img.get('system:time_start')))

    roll = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
              .select('precipitation')
              .filter(ee.Filter.date(start_date, end_date))
              .map(rolling30))

    mean30 = roll.mean()
    std30  = roll.reduce(ee.Reducer.stdDev())
    spi    = roll.map(lambda img: img.subtract(mean30).divide(std30.add(1e-6)).rename('SPI30'))
    return spi


def extract_features(lat, lon, year):
    """
    Extract kharif-season features for one (lat, lon, year) combination.
    Returns a dict of scalar values.
    """
    pt    = ee.Geometry.Point([lon, lat])
    aoi   = pt.buffer(BUFFER_M)
    start = f'{year}-{KHARIF_START_MD}'
    end   = f'{year}-{KHARIF_END_MD}'

    def sample(img, band=None):
        """Return mean value over the AOI."""
        img_b = img.select(band) if band else img
        val = img_b.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=SCALE_M,
            maxPixels=1e7,
        )
        return val

    # ─── CHIRPS ───────────────────────────────────────────────────────────────
    chirps_daily = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                      .select('precipitation')
                      .filter(ee.Filter.date(start, end)))
    cumRain = chirps_daily.sum().reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7)

    # 30-day SPI at season end
    spi_col = chirps_spi30(start, end)
    spi_val = (spi_col.mean().reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7))

    # Dry spell: days with precipitation < 2 mm
    dry_days = (chirps_daily
                  .map(lambda img: img.lt(2).rename('dry'))
                  .sum()
                  .reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7))

    # ─── ERA5-Land ────────────────────────────────────────────────────────────
    era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
              .select('temperature_2m_max')
              .filter(ee.Filter.date(start, end)))
    tmax_mean = era5.mean().reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7)
    tmax_anom_img = era5.mean().subtract(era5_base)
    tmax_anom_val = tmax_anom_img.reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7)

    # ─── ERA5 Soil Moisture (proxy for SMAP) ─────────────────────────────────
    era5_full = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                   .select('volumetric_soil_water_layer_1')
                   .filter(ee.Filter.date(start, end)))
    sm_base = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                 .select('volumetric_soil_water_layer_1')
                 .filter(ee.Filter.calendarRange(6, 10, 'month'))
                 .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                 .mean())
    sm_mean    = era5_full.mean().reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7)
    sm_anom_v  = era5_full.mean().subtract(sm_base).reduceRegion(ee.Reducer.mean(), aoi, SCALE_M, maxPixels=1e7)

    # ─── MODIS NDVI ───────────────────────────────────────────────────────────
    ndvi_col = (ee.ImageCollection('MODIS/061/MOD13Q1')
                  .select('NDVI')
                  .filter(ee.Filter.date(start, end))
                  .map(lambda img: img.multiply(0.0001)))
    ndvi_peak = ndvi_col.max().reduceRegion(ee.Reducer.mean(), aoi, 500, maxPixels=1e7)
    ndvi_mean = ndvi_col.mean().reduceRegion(ee.Reducer.mean(), aoi, 500, maxPixels=1e7)
    ndvi_anom = (ndvi_col.mean().subtract(modis_ndvi_base)
                          .reduceRegion(ee.Reducer.mean(), aoi, 500, maxPixels=1e7))

    # ─── MODIS LST ────────────────────────────────────────────────────────────
    lst_col = (ee.ImageCollection('MODIS/061/MOD11A2')
                 .select('LST_Day_1km')
                 .filter(ee.Filter.date(start, end))
                 .map(lambda img: img.multiply(0.02).subtract(273.15)))
    lst_mean = lst_col.mean().reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=1e7)
    lst_anom = (lst_col.mean().subtract(modis_lst_base)
                        .reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=1e7))

    # ─── VHI: Vegetation Health Index = 0.5*VCI + 0.5*TCI ───────────────────
    # (needs multi-year min/max — approximate with baseline period)
    ndvi_min_base = (ee.ImageCollection('MODIS/061/MOD13Q1')
                       .select('NDVI')
                       .filter(ee.Filter.calendarRange(6, 10, 'month'))
                       .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                       .min().multiply(0.0001))
    ndvi_max_base = (ee.ImageCollection('MODIS/061/MOD13Q1')
                       .select('NDVI')
                       .filter(ee.Filter.calendarRange(6, 10, 'month'))
                       .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                       .max().multiply(0.0001))
    lst_min_base  = (ee.ImageCollection('MODIS/061/MOD11A2')
                       .select('LST_Day_1km')
                       .filter(ee.Filter.calendarRange(6, 10, 'month'))
                       .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                       .min().multiply(0.02).subtract(273.15))
    lst_max_base  = (ee.ImageCollection('MODIS/061/MOD11A2')
                       .select('LST_Day_1km')
                       .filter(ee.Filter.calendarRange(6, 10, 'month'))
                       .filter(ee.Filter.date('2016-01-01', '2026-01-01'))
                       .max().multiply(0.02).subtract(273.15))
    ndvi_now = ndvi_col.mean()
    lst_now  = lst_col.mean()
    vci = ndvi_now.subtract(ndvi_min_base).divide(ndvi_max_base.subtract(ndvi_min_base).add(1e-6))
    tci = lst_max_base.subtract(lst_now).divide(lst_max_base.subtract(lst_min_base).add(1e-6))
    vhi = vci.multiply(0.5).add(tci.multiply(0.5)).rename('VHI')
    vhi_val = vhi.reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=1e7)

    # ─── Assemble and evaluate ────────────────────────────────────────────────
    result = ee.Dictionary({
        'cumRain_mm':   cumRain.get('precipitation'),
        'SPI30_mean':   spi_val.get('SPI30'),
        'drySpellDays': dry_days.get('dry'),
        'tmax_K_mean':  tmax_mean.get('temperature_2m_max'),
        'tmax_anom_K':  tmax_anom_val.get('temperature_2m_max'),
        'SM_mean':      sm_mean.get('volumetric_soil_water_layer_1'),
        'SM_anom':      sm_anom_v.get('volumetric_soil_water_layer_1'),
        'NDVI_peak':    ndvi_peak.get('NDVI'),
        'NDVI_mean':    ndvi_mean.get('NDVI'),
        'NDVI_anom':    ndvi_anom.get('NDVI'),
        'LST_mean_C':   lst_mean.get('LST_Day_1km'),
        'LST_anom_C':   lst_anom.get('LST_Day_1km'),
        'VHI_mean':     vhi_val.get('VHI'),
    })
    return result.getInfo()

print('Helper functions defined.')

In [ ]:
# ── Main extraction loop ──────────────────────────────────────────────────────
# Checkpoint: resume from where we left off if the output CSV already exists
if OUT_CSV.exists():
    existing = pd.read_csv(OUT_CSV)
    done_keys = set(zip(existing['revenue_circle'], existing['year']))
    print(f'Resuming — {len(existing)} rows already done.')
else:
    existing = pd.DataFrame()
    done_keys = set()

rows = []
total = len(coords_df) * len(KHARIF_YEARS)
done  = len(done_keys)

for _, site in coords_df.iterrows():
    rc     = site['revenue_circle']
    taluka = site['taluka']
    lat    = site['lat']
    lon    = site['lon']

    for year in KHARIF_YEARS:
        if (rc, year) in done_keys:
            continue

        done += 1
        try:
            feats = extract_features(lat, lon, year)
            row = {'taluka': taluka, 'revenue_circle': rc,
                   'lat': lat, 'lon': lon, 'year': year,
                   'geocode_flag': site['geocode_flag']}
            row.update(feats)
            rows.append(row)
            status = '✓'
        except Exception as e:
            rows.append({'taluka': taluka, 'revenue_circle': rc,
                         'lat': lat, 'lon': lon, 'year': year,
                         'geocode_flag': site['geocode_flag'], 'error': str(e)[:120]})
            status = f'✗ {str(e)[:60]}'

        if done % 10 == 0 or done == total:
            # Checkpoint save every 10 rows
            df_partial = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
            df_partial.to_csv(OUT_CSV, index=False)

        pct = done / total * 100
        print(f'[{done:03d}/{total}] {pct:.0f}% — {taluka}/{rc} {year}: {status}', flush=True)

# Final save
df_out = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
df_out.to_csv(OUT_CSV, index=False)
print(f'\nDone. {len(df_out)} rows saved to {OUT_CSV}')

In [ ]:
# ── Join satellite features with PMFBY IU claim data ────────────────────────
feats  = pd.read_csv('data/processed/yavatmal_rc_features_v1.csv')
pmfby  = pd.read_csv('data/raw/pmfby_yavatmal_iu_kharif.csv')

# NOTE: at IU-level, 'claim_total' is a farmer count (not rupees).
# claim_ratio = fraction of enrolled farmers who received any payout.
for c in ['farmers','claim_total','sum_insured_rs']:
    pmfby[c] = pd.to_numeric(pmfby[c], errors='coerce')

pmfby_rc = (pmfby.groupby(['taluka','revenue_circle','year'])
                 .agg(farmers_total=('farmers','sum'),
                      claim_farmers=('claim_total','sum'),
                      sum_insured=('sum_insured_rs','sum'))
                 .reset_index())
# Primary: payout_rate = farmers who got a claim / total enrolled farmers
pmfby_rc['claim_ratio'] = (pmfby_rc['claim_farmers']
                            / pmfby_rc['farmers_total'].replace(0, np.nan))

# Normalise revenue_circle names (strip whitespace)
feats['revenue_circle']    = feats['revenue_circle'].str.strip()
pmfby_rc['revenue_circle'] = pmfby_rc['revenue_circle'].str.strip()

merged = feats.merge(pmfby_rc[['taluka','revenue_circle','year','claim_ratio','farmers_total','claim_farmers','sum_insured']],
                     on=['taluka','revenue_circle','year'], how='left')

merged.to_csv('data/processed/yavatmal_rc_model_ready_v1.csv', index=False)
n_with_pmfby = merged['claim_ratio'].notna().sum()
print(f'Merged dataset: {len(merged)} rows, {n_with_pmfby} with PMFBY claim_ratio')
merged[['taluka','revenue_circle','year','claim_ratio','VHI_mean','SPI30_mean','SM_anom','NDVI_anom']].head(20)

In [ ]:
# ── Quick EDA — feature × claim_ratio correlations ───────────────────────────
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

model_df = pd.read_csv('data/processed/yavatmal_rc_model_ready_v1.csv').dropna(subset=['claim_ratio'])
feat_cols = ['cumRain_mm','SPI30_mean','drySpellDays','tmax_anom_K',
             'SM_anom','NDVI_peak','NDVI_anom','LST_anom_C','VHI_mean']

corrs = {}
for f in feat_cols:
    if f in model_df.columns:
        valid = model_df[[f,'claim_ratio']].dropna()
        if len(valid) > 5:
            r, p = spearmanr(valid[f], valid['claim_ratio'])
            corrs[f] = (r, p)

corr_df = (pd.DataFrame(corrs, index=['rho','p']).T
             .sort_values('rho')
             .assign(sig=lambda d: d['p'].map(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')))

print('Spearman ρ (feature vs claim_ratio):')
print(corr_df[['rho','p','sig']].round(3).to_string())

fig, ax = plt.subplots(figsize=(8,4))
colors = ['#d73027' if r < 0 else '#1a9850' for r in corr_df['rho']]
ax.barh(corr_df.index, corr_df['rho'], color=colors)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Spearman ρ with claim_ratio')
ax.set_title('Feature correlations — Yavatmal revenue circles')
plt.tight_layout()
plt.savefig('vidarbha_outputs/yavatmal_rc_correlations.png', dpi=150)
plt.show()
print('Plot saved.')